# Fused Transformer Kernels in Triton

Runs the correctness tests and the benchmark on a GPU.

**First:** Runtime → Change runtime type → **T4 GPU** (or A100 if you have Colab Pro) → Save.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
!git clone -q https://github.com/riddhixraina/triton-fused-kernels
%cd triton-fused-kernels
!pip install -q -r requirements.txt

/content/triton-fused-kernels


## 1. Correctness: every kernel must match the PyTorch reference

In [5]:
!git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 3), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 2.26 KiB | 2.26 MiB/s, done.
From https://github.com/riddhixraina/triton-fused-kernels
   49f4a8b..e66ec04  main       -> origin/main
Updating 49f4a8b..e66ec04
Fast-forward
 colab.ipynb | 227 ++++++++++++++++++++++++++++++++++++------------------------
 kernels.py  |  54 +++++++++------
 2 files changed, 170 insertions(+), 111 deletions(-)


In [6]:
!pytest -q

.....................................                                    [100%]
37 passed in 22.15s


## 2. Benchmark against PyTorch eager and torch.compile

T4 GPUs have no bfloat16, so use fp16 there. On an A100, also try `--dtype bf16` and `--model llama3-8b`.

In [7]:
!python bench.py --dtype fp16 --csv results_fp16.csv

GPU: NVIDIA A100-SXM4-40GB   peak bandwidth: 1555 GB/s
Model shapes: qwen2.5-7b (hidden 3584, intermediate 18944), dtype fp16

op            tokens   eager ms  compile ms  triton ms   speedup      GB/s   % peak
rmsnorm            1     0.0451      0.0092     0.0072     6.29x         2       0%
add+rmsnorm        1     0.0502      0.0082     0.0082     6.12x         4       0%
silu_and_mul       1     0.0102      0.0072     0.0072     1.43x        16       1%
rmsnorm           16     0.0451      0.0072     0.0072     6.29x        32       2%
add+rmsnorm       16     0.0594      0.0082     0.0082     7.25x        56       4%
silu_and_mul      16     0.0174      0.0092     0.0082     2.13x       222      14%
rmsnorm          128     0.0512      0.0082     0.0072     7.14x       256      16%
add+rmsnorm      128     0.0696      0.0102     0.0102     6.80x       358      23%
silu_and_mul     128     0.0358      0.0195     0.0184     1.94x       789      51%
rmsnorm          512     0.0748  

In [8]:
!python bench.py --dtype bf16 --csv results_bf16.csv

GPU: NVIDIA A100-SXM4-40GB   peak bandwidth: 1555 GB/s
Model shapes: qwen2.5-7b (hidden 3584, intermediate 18944), dtype bf16

op            tokens   eager ms  compile ms  triton ms   speedup      GB/s   % peak
rmsnorm            1     0.0471      0.0102     0.0082     5.75x         2       0%
add+rmsnorm        1     0.0522      0.0082     0.0082     6.37x         4       0%
silu_and_mul       1     0.0102      0.0072     0.0072     1.43x        16       1%
rmsnorm           16     0.0461      0.0072     0.0072     6.43x        32       2%
add+rmsnorm       16     0.0614      0.0082     0.0082     7.50x        56       4%
silu_and_mul      16     0.0174      0.0092     0.0092     1.89x       197      13%
rmsnorm          128     0.0532      0.0082     0.0082     6.50x       224      14%
add+rmsnorm      128     0.0717      0.0102     0.0092     7.78x       398      26%
silu_and_mul     128     0.0358      0.0195     0.0195     1.84x       748      48%
rmsnorm          512     0.0758  

In [9]:
!python bench.py --dtype bf16 --model llama3-8b --csv results_llama_bf16.csv

GPU: NVIDIA A100-SXM4-40GB   peak bandwidth: 1555 GB/s
Model shapes: llama3-8b (hidden 4096, intermediate 14336), dtype bf16

op            tokens   eager ms  compile ms  triton ms   speedup      GB/s   % peak
rmsnorm            1     0.0461      0.0102     0.0072     6.43x         2       0%
add+rmsnorm        1     0.0502      0.0082     0.0072     7.00x         5       0%
silu_and_mul       1     0.0102      0.0072     0.0072     1.43x        12       1%
rmsnorm           16     0.0471      0.0072     0.0072     6.57x        37       2%
add+rmsnorm       16     0.0625      0.0092     0.0072     8.71x        73       5%
silu_and_mul      16     0.0164      0.0082     0.0082     2.00x       168      11%
rmsnorm          128     0.0532      0.0082     0.0082     6.50x       256      16%
add+rmsnorm      128     0.0727      0.0102     0.0102     7.10x       410      26%
silu_and_mul     128     0.0307      0.0164     0.0154     2.00x       717      46%
rmsnorm          512     0.0799   